# Разбор A. Область прогноза и идентифицируемость кинетической модели

Продолжаем анализ реальных оценок кофеина из [Table 4](https://doi.org/10.1186/s40064-015-0809-x). Перед расчётом сформулируйте критерий: прогноз проверяется на исключённой концентрации, а решение о переносе за 0.02–0.10 M требует нового эксперимента. Table 5 статьи содержит авторскую экстраполяцию на 30 °C; её нельзя добавлять как независимые измерения для обучения или проверки.

In [2]:
from pathlib import Path
import sys,json,math,csv
import numpy as np
from scipy import stats
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,workspace,write_json,init_dossier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=workspace('M09-L02','case')
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=150,bbox_inches='tight')
    display(fig)
    plt.close(fig)


Хорошая подгонка ещё не отвечает, как модель поведёт себя на неиспользованной концентрации. Поэтому я исключаю одну концентрацию, заново оцениваю коэффициенты по остальным и сравниваю прогноз с исключённым значением. Важно именно заново оценить модель: остаток первоначальной подгонки уже использовал проверяемую точку.

Формула e_i/(1−h_i) даёт удалённый остаток линейной модели. В коде я дополнительно проверяю её явным повторным fit. Величина h_i отражает положение входа относительно плана: точка с большим влиянием может иметь маленький обычный остаток и заметно большую ошибку исключённого прогноза. Эта связь объясняет практическую пользу проверки, а не просто добавляет ещё одну метрику.

In [3]:
with (DATA/'caffeine_hydroxide_table4.csv').open(encoding='utf-8',newline='') as f:
    rows=list(csv.DictReader(f))
x=np.array([float(r['NaOH_mol_L']) for r in rows])
y=np.array([float(r['kexp_min_inverse']) for r in rows])
# Два столбца соответствуют свободному члену и зависимости от [OH-].
X=np.column_stack([np.ones(len(x)),x])
beta=np.linalg.lstsq(X,y,rcond=None)[0]
residual=y-X@beta
df=len(y)-X.shape[1]
inverse=np.linalg.inv(X.T@X)
s2=float(residual@residual/df)
covariance=s2*inverse
critical=stats.t.ppf(.975,df)
ci=np.column_stack([beta-critical*np.sqrt(np.diag(covariance)),beta+critical*np.sqrt(np.diag(covariance))])
print('intercept / min^-1; slope / L mol^-1 min^-1:',beta)
print('Conditional OLS intervals, NOT measured replicate uncertainty:',ci)


intercept / min^-1; slope / L mol^-1 min^-1: [0.0013 0.695 ]
Conditional OLS intervals, NOT measured replicate uncertainty: [[-0.02116501  0.02376501]
 [ 0.3563273   1.0336727 ]]


Диагональ матрицы H=X(XᵀX)⁻¹Xᵀ даёт leverage. Для линейного OLS остаток при исключении строки равен eᵢ/(1−hᵢ). Проверим эту формулу независимым переобучением пяти моделей, исключая целую концентрацию.

In [4]:
leverage=np.einsum('ij,jk,ik->i',X,inverse,X)
deleted_residual=residual/(1-leverage)
explicit=[]
for i in range(len(y)):
    keep=np.arange(len(y))!=i
    fitted=np.linalg.lstsq(X[keep],y[keep],rcond=None)[0]
    explicit.append(y[i]-X[i]@fitted)
assert np.allclose(deleted_residual,explicit,atol=1e-12)
rmse=float(np.sqrt(np.mean(deleted_residual**2)))
print('Leave-one-concentration-out RMSE / min^-1:',rmse)
query=np.array([.001,.03,.12])
Q=np.column_stack([np.ones(len(query)),query])
prediction=Q@beta
variance_mean=s2*np.einsum('ij,jk,ik->i',Q,inverse,Q)
half_width=critical*np.sqrt(s2+variance_mean)
display([{'NaOH_mol_L':float(c),'kexp':float(v),'conditional_prediction_interval':[float(v-w),float(v+w)],
          'domain':'interpolation' if min(x)<=c<=max(x) else 'extrapolation'}
         for c,v,w in zip(query,prediction,half_width)])
fig,ax=plt.subplots()
ax.errorbar(query,prediction,yerr=half_width,fmt='o',label='Conditional OLS prediction interval')
ax.scatter(x,y,label='Published estimates');ax.axhline(0,color='gray')
ax.set(xlabel='NaOH / mol L^-1',ylabel='k_exp / min^-1');ax.legend(fontsize=7)
publish_plot(fig,'prediction_domain.png')


Leave-one-concentration-out RMSE / min^-1: 0.008266125242857893


[{'NaOH_mol_L': 0.001,
  'kexp': 0.0019950000000000046,
  'conditional_prediction_interval': [-0.02882421571058901,
   0.03281421571058902],
  'domain': 'extrapolation'},
 {'NaOH_mol_L': 0.03,
  'kexp': 0.022150000000000006,
  'conditional_prediction_interval': [-0.0034192281362463925,
   0.047719228136246405],
  'domain': 'interpolation'},
 {'NaOH_mol_L': 0.12,
  'kexp': 0.08470000000000001,
  'conditional_prediction_interval': [0.05366013431803378,
   0.11573986568196624],
  'domain': 'extrapolation'}]

<Figure size 640x480 with 1 Axes>

Отрицательная нижняя граница показывает ограничение линейной модели и малой выборки; её нельзя молча обрезать и объявить физическим интервалом. Для позитивной модели нужен обоснованный закон ошибки и сравнение на новых данных. По этой таблице нельзя заново оценить Eₐ: она содержит одну температуру. Следующий опыт должен включать несколько температур, независимые приготовления, quench, проверку извлечения и отклика, а также случайный порядок анализа.

In [5]:
result={'kind':'calculated','analysis_scope':'published_data_reanalysis','source_doi':'10.1186/s40064-015-0809-x','slope_L_mol_inverse_min_inverse':float(beta[1]),'raw_time_series_refit':'NOT_RUN','scientific_review':'NOT_RUN'}
result['LOOCV_RMSE_min_inverse']=rmse


У прогнозного интервала появляется дополнительная остаточная дисперсия s² поверх неопределённости среднего отклика. Поэтому он шире интервала среднего при тех же предположениях. Я отдельно отмечаю интерполяцию и выход за измеренный диапазон: линейная формула сама продолжится куда угодно, но экспериментального основания за границами не добавится. Если нижняя граница отрицательна, я сохраняю этот сигнал ограничения модели и планирую проверку вместо молчаливого обрезания.

In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
receipt=append_record(init_dossier(),'M09-L02','A',result,ROOT,inputs=['practicuum/shared/data/caffeine_hydroxide_table4.csv', 'practicuum/shared/data/caffeine_kinetics_provenance.json', 'practicuum/shared/data/source-responses/caffeine_kinetics.xml', 'practicuum/shared/data/source-responses/statin_stability_verified.xml'])
print(receipt)


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\reasoning-full-20261120\workspace\my_research\evidence\M09-L02\20261120T105405216635Z-A-d47153c38209421c84d92dcf3ab6f373.json


# M09. Как наставник читает первичную работу

Пока следующая вычислительная задача ещё не выбрана, вернёмся к источнику: **Caffeine in surface and wastewaters in Barbados, West Indies (2015)**. [Локальный полный текст](../../../../../shared/data/source-responses/caffeine_kinetics.xml), [DOI](https://doi.org/10.1186/s40064-015-0809-x). Откройте **Table 4 и модель кинетики; перенос на работу о статинах**.

Проверьте единицы и условия опубликованных констант кофеина. Назовите его роль контрольного ксантина.

Наставник начинает с вопроса «что здесь непосредственно наблюдали?». Найдите одно такое наблюдение и его место в источнике. Затем рядом запишите трактовку авторов и ещё одно возможное объяснение. Это учебный приём чтения, а не реконструкция мыслей авторов. Если численного ряда нет, вывод нельзя усиливать придуманными точками.

Перед вычислением назовите объект, условия, величину и единицы. После вычисления сравните ответ с исходным утверждением. При расхождении сначала проверьте форму вещества, основу концентрации и преобразования таблицы; только затем обсуждайте модель. Смысл операции — получить проверяемый аргумент для следующего решения досье.

Теперь перенесите приём на [самостоятельную мини-рецензию B](../steps/03_article_review.md). Для проверки собственного вывода используйте вопрос: **Может ли одинаковое обозначение k оправдать сравнение констант с разной размерностью?**